# RAG v2 — Alert-Triggered Grounded Explanations (step by step)

Rebuilds §4.4–4.10 of the revised paper. The v2 changes this notebook walks through:

1. **Corpus expansion** — two wearable-relevant guidance documents added (EHRA 2022
   digital-devices guide; 2023 ACC/AHA AF guideline) after the v1 audit showed the
   original guidelines matched almost none of the alerts.
2. **Query semantics fixed** — z-scores vs each subject's *non-flagged* windows
   (not, as in v1, vs the population of flagged windows), evidence-tied character
   phrases, per-subject (online-capable) reference.
3. **Raw pre-canonicalization text preserved** — citation accuracy before AND after repair.
4. **Judges validated** on a 200-item corruption benchmark before use.
5. **Labeled-event evaluation** — 148 true events, labels never in the queries.

Caches: `chroma_db_v2` (skip re-embed if populated, as in v1), `generation_v2.jsonl`
(resume-safe generation), judge CSVs. Delete them for a full rebuild (~3 h GPU +
OpenRouter key for the API judge).

## 1. Imports & configuration

In [1]:
import difflib, json, re, sys, time
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd(); OUT = ROOT/"outputs_v2"
GEN_JSONL = OUT/"generation_v2.jsonl"
LLM_MODEL, ALT_MODEL = "qwen3.5:9b", "llama3.1:8b"
JUDGE_CANDIDATES = ["llama3.1:8b", "gemma4:e4b"]   # gpt-oss excluded per author decision
API_MODEL = "deepseek/deepseek-v4-flash-0731"
CHANNELS = ["ecg","resp","bvp","wrist_eda","wrist_temp"]
TOPIC_PHRASES = {
    "ecg": "electrocardiogram rhythm irregularity heart rate variability arrhythmia ectopic beats atrial fibrillation",
    "resp": "respiration rate breathing pattern tachypnea bradypnea ventilation",
    "bvp": "photoplethysmography pulse waveform amplitude perfusion signal quality motion artifact atrial fibrillation screening",
    "wrist_eda": "electrodermal activity skin conductance sympathetic stress arousal sweat response",
    "wrist_temp": "skin temperature thermal perfusion vasomotor ambient temperature sensor effects"}
ECG_TOPICS = TOPIC_PHRASES["ecg"] + " ventricular tachycardia conduction abnormality repolarization ST changes myocardial infarction hypertrophy"
print("config OK")

config OK


## 2. Corpus — Tier-1 v1 (4 guideline PDFs) + Tier-1 v2 (2 wearable-relevant) + Tier-2 (200 OA articles)

In [2]:
import fitz
TIER1_DIR = ROOT/"Dataset/RAG corpus (medical literatureguidelines)"
TIER1_V2 = ROOT/"Dataset/Tier1_v2"; TIER2_DIR = ROOT/"Dataset/Tier2_literature"

docs = []
for pdf in sorted(TIER1_DIR.glob("*.pdf")):
    d = fitz.open(pdf); text = "\n".join(pg.get_text() for pg in d); d.close()
    docs.append({"source": pdf.stem, "tier":"tier1", "tier1_v":"v1", "text": text})
    print(f"  tier1/v1  {pdf.stem[:55]:55s} {len(text):>9,} chars")
for t in sorted(TIER1_V2.glob("*.txt")):
    docs.append({"source": t.stem, "tier":"tier1", "tier1_v":"v2", "text": t.read_text(encoding="utf-8")})
    print(f"  tier1/v2  {t.stem[:55]:55s} {len(docs[-1]['text']):>9,} chars")
for bdir in sorted(TIER2_DIR.iterdir()):
    if not bdir.is_dir(): continue
    mds = sorted(bdir.glob("*.md"))
    for m in mds:
        docs.append({"source": m.stem, "tier":"tier2", "bucket": bdir.name, "text": m.read_text(encoding="utf-8")})
    print(f"  tier2     {bdir.name:55s} {len(mds):>5} articles")
print(f"\nTOTAL: {len(docs)} documents")

  tier1/v1  2017 ACCAHAHRS — Evaluation of Patients with Syncope      361,430 chars


  tier1/v1  2017 AHA_ACC_HRS — Management of Ventricular Arrhythmia   687,839 chars
  tier1/v1  ESC 2018 — Guidelines for the Diagnosis and Management    359,141 chars


  tier1/v1  ESC 2022 — Guidelines for Ventricular Arrhythmias & SCD   725,258 chars
  tier1/v2  accaha2023_af_guideline                                   771,946 chars
  tier1/v2  ehra2022_digital_devices_arrhythmias                      140,248 chars
  tier2     01_ppg_arrhythmia                                          50 articles
  tier2     02_ppg_signal_quality                                      25 articles
  tier2     03_ecg_anomaly_ml                                          45 articles
  tier2     04_wearable_stress                                         35 articles
  tier2     05_continuous_monitoring                                   25 articles
  tier2     06_biosignal_methods                                       20 articles

TOTAL: 206 documents


## 3. Chunk (500 words / 50 overlap, v1-identical) and embed into `chroma_db_v2`

In [3]:
def chunk_text(text, chunk_words=500, overlap=50):
    words = text.split()
    if len(words) <= chunk_words: return [text]
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start+chunk_words])); start += chunk_words - overlap
    return chunks

all_chunks = []
for doc in docs:
    for i, ch in enumerate(chunk_text(doc["text"])):
        all_chunks.append({"text": ch, "source": doc["source"], "tier": doc["tier"],
                           "tier1_v": doc.get("tier1_v",""), "bucket": doc.get("bucket",""), "chunk_idx": i})
print(f"{len(all_chunks):,} chunks (mean {np.mean([len(c['text'].split()) for c in all_chunks]):.0f} words)")

import chromadb
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
col = chromadb.PersistentClient(path=str(ROOT/"chroma_db_v2")).get_or_create_collection("medical_corpus_v2")
if col.count() == 0:
    t0 = time.time()
    embs = []
    for i in range(0, len(all_chunks), 64):
        embs.extend(embedder.encode([c["text"] for c in all_chunks[i:i+64]]).tolist())
    col.add(embeddings=embs, documents=[c["text"] for c in all_chunks],
           metadatas=[{k:v for k,v in c.items() if k!="text"} for c in all_chunks],
           ids=[f"chunk_{i}" for i in range(len(all_chunks))])
    print(f"embedded {len(all_chunks):,} chunks in {time.time()-t0:.0f}s")
else:
    print(f"collection already populated ({col.count():,} chunks) — skipping embed (v1 convention)")

5,045 chunks (mean 489 words)


C:\Users\tanvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


collection already populated (5,045 chunks) — skipping embed (v1 convention)


## 4. Retrieval (dense + source-diversity) — sanity test

In [4]:
def retrieve(query, top_k=5, pool=20, max_per_source=1):
    res = col.query(query_embeddings=embedder.encode([query]).tolist(), n_results=pool,
                    include=["documents","metadatas","distances"])
    metas = res["metadatas"][0]
    picked, counts = [], {}
    for i, m in enumerate(metas):
        cnt = counts.get(m["source"], 0)
        if cnt >= max_per_source: continue
        picked.append(i); counts[m["source"]] = cnt+1
        if len(picked) == top_k: break
    if len(picked) < top_k:
        for i in range(len(metas)):
            if i not in picked: picked.append(i)
            if len(picked) == top_k: break
    return {"sources":[metas[i]["source"] for i in picked],
            "tiers":[metas[i]["tier"] for i in picked],
            "docs":[res["documents"][0][i] for i in picked],
            "dists":[res["distances"][0][i] for i in picked]}

for q in ["How does PPG detect atrial fibrillation?",
          "What causes false alarms in wearable heart rate monitors?",
          "How is stress detected from electrodermal activity?"]:
    r = retrieve(q, top_k=3)
    print("QUERY:", q)
    for s, t, d in zip(r["sources"], r["tiers"], r["dists"]):
        print(f"   [{t}] {s[:70]} (d={d:.3f})")
    print()

QUERY: How does PPG detect atrial fibrillation?
   [tier1] ehra2022_digital_devices_arrhythmias (d=0.677)
   [tier2] PMC12635274_fibricheck_detection_capabilities_for_atrial_fibrillation_ (d=0.702)
   [tier2] PMC12925684_diagnostic_performance_of_two_commercially_available_ppgba (d=0.712)

QUERY: What causes false alarms in wearable heart rate monitors?
   [tier2] PMC13027176_the_use_of_digital_devices_in_the_management_of_athletes_w (d=0.726)
   [tier2] PMC12986385_atrial_fibrillation_and_cognitive_decline_a_systematic_rev (d=0.749)
   [tier1] ehra2022_digital_devices_arrhythmias (d=0.750)

QUERY: How is stress detected from electrodermal activity?
   [tier2] PMC13211236_electrodermal_temperatureadjusted_electrodermal_activity_e (d=0.653)
   [tier2] PMC13076599_supervised_information_gainbased_feature_selection_for_mul (d=0.675)
   [tier2] PMC12608435_shortterm_detection_of_dynamic_stress_levels_in_exergaming (d=0.679)



## 5. The 398 alerts — archived flags mapped exactly onto the v2 feature matrix

In [5]:
z = np.load(OUT/"cache/ppgdalia.npz", allow_pickle=True)
X_all, subj_all = z["X"], z["subject"]
cols = [f"{ch}__{fn}" for ch in CHANNELS
        for fn in ["mean","std","min","max","ptp","median","skew","kurt","p25","p75","up_ratio","roughness"]]
df_all = pd.DataFrame(X_all, columns=cols); df_all["subject"] = subj_all
df_all["window_idx"] = np.arange(len(df_all))

flagged = pd.read_parquet(ROOT/"outputs_v1_archive/flagged_windows.parquet")
df_all["flag_if"] = False; df_all["flag_lof"] = False
offsets, off = {}, 0
for s in sorted(df_all.subject.unique()):
    offsets[s] = off; off += int((df_all.subject==s).sum())
for _, r in flagged.iterrows():
    g = offsets[int(r["subject"])] + int(r["window_idx"])
    df_all.loc[g, "flag_if"] = bool(r["flag_if"]); df_all.loc[g, "flag_lof"] = bool(r["flag_lof"])
print(f"IF {int(df_all.flag_if.sum())}, LOF {int(df_all.flag_lof.sum())}, "
      f"both {int((df_all.flag_if & df_all.flag_lof).sum())}, "
      f"union {int((df_all.flag_if | df_all.flag_lof).sum())}  (v1 archived: 216/216/34/398)")

IF 216, LOF 216, both 34, union 398  (v1 archived: 216/216/34/398)


## 6. Query builder v2 — correct reference class + evidence-tied character

In [6]:
def subject_reference(df, subject):
    sub = df[df.subject == subject]
    normal = sub[(~sub.flag_if) & (~sub.flag_lof)]          # the detector's normal population
    ref = {}
    for ch in CHANNELS:
        for feat in ["mean","kurt","ptp"]:
            c = f"{ch}__{feat}"
            ref[c] = (normal[c].mean(), normal[c].std(ddof=0) or 1e-9)
    return ref

def build_query_v2(row, ref):
    zs = {ch: (row[f"{ch}__mean"]-ref[f"{ch}__mean"][0]) / ref[f"{ch}__mean"][1] for ch in CHANNELS}
    kzs = {ch: (row[f"{ch}__kurt"]-ref[f"{ch}__kurt"][0]) / ref[f"{ch}__kurt"][1] for ch in CHANNELS}
    pzs = {ch: (row[f"{ch}__ptp"] -ref[f"{ch}__ptp"][0])  / ref[f"{ch}__ptp"][1]  for ch in CHANNELS}
    top2 = sorted(CHANNELS, key=lambda ch: -abs(zs[ch]))[:2]
    if row.flag_if and row.flag_lof: parts = ["Biosignal window flagged by both anomaly detectors (Isolation Forest and LOF)."]
    elif row.flag_if:                parts = ["Biosignal window flagged by the Isolation Forest anomaly detector."]
    else:                            parts = ["Biosignal window flagged by the LOF anomaly detector."]
    shape = max(max(kzs[ch], pzs[ch]) for ch in top2); shift = max(abs(zs[ch]) for ch in top2)
    if shape > 1.5:    parts.append("Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline).")
    elif shift > 1.5:  parts.append("Deviating channels show a sustained level shift from this subject's baseline.")
    elif shift > 1.0:  parts.append("Deviating channels show a moderate shift from this subject's baseline.")
    else:              parts.append("Deviating channels are only mildly unusual vs this subject's baseline.")
    for ch in top2:
        parts.append(f"{'elevated' if zs[ch]>0 else 'reduced'} {ch} (z={zs[ch]:+.1f} vs subject baseline, mean={row[f'{ch}__mean']:.2f})")
    parts.append("Relevant topics: " + " ".join(TOPIC_PHRASES[ch] for ch in top2) + ".")
    parts.append("Other readings: " + ", ".join(f"{ch} mean={row[f'{ch}__mean']:.2f}" for ch in CHANNELS if ch not in top2) + ".")
    return " ".join(parts), {f"z_{ch}": round(zs[ch],3) for ch in CHANNELS}

union = df_all[df_all.flag_if | df_all.flag_lof]
for _, row in union.head(3).iterrows():
    q, zs = build_query_v2(row, subject_reference(df_all, row.subject))
    print(f"S{int(row.subject)} w{int(row.window_idx)}: {q[:220]}...\n")

S1 w1: Biosignal window flagged by the LOF anomaly detector. Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline). elevated bvp (z=+0.8 vs subject baseline, mean=...

S1 w6: Biosignal window flagged by the LOF anomaly detector. Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline). elevated resp (z=+1.0 vs subject baseline, mean...

S1 w13: Biosignal window flagged by the LOF anomaly detector. Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline). reduced wrist_eda (z=-0.6 vs subject baseline, ...



## 7. Sanity check — the corrected z-metric separates stress from baseline on WESAD

In [7]:
from scipy.stats import mannwhitneyu
w = np.load(OUT/"cache/wesad.npz", allow_pickle=True)
stress_top2, baseline_top2 = [], []
for s in w["subjects"]:
    s = int(s); X, y = w[f"X_{s}"], w[f"y_{s}"]
    dfw = pd.DataFrame(X, columns=cols); base = dfw[y==1]
    ref = {ch: (base[f"{ch}__mean"].mean(), base[f"{ch}__mean"].std(ddof=0) or 1e-9) for ch in CHANNELS}
    for mask, acc in [(y==2, stress_top2), (y==1, baseline_top2)]:
        for _, r in dfw[mask].iterrows():
            zz = [abs((r[f"{ch}__mean"]-ref[ch][0])/ref[ch][1]) for ch in CHANNELS]
            acc.append(sorted(zz)[-2:])
st, bt = np.array(stress_top2).max(axis=1), np.array(baseline_top2).max(axis=1)
u, p = mannwhitneyu(st, bt, alternative="greater")
print(f"stress max|z| mean {st.mean():.1f} vs baseline {bt.mean():.1f} — Mann-Whitney p = {p:.3g}")

stress max|z| mean 43.7 vs baseline 1.5 — Mann-Whitney p = 2.54e-138


## 8. Build all 398 queries + retrieval (loads the jsonl if already built)

In [8]:
p = OUT/"alerts_retrieval_v2.jsonl"
if p.exists():
    alerts = [json.loads(l) for l in open(p, encoding="utf-8")]
    print(f"loaded {len(alerts)} cached retrievals")
else:
    alerts = []
    for _, row in union.iterrows():
        q, _ = build_query_v2(row, subject_reference(df_all, row.subject))
        r = retrieve(q)
        alerts.append({"subject": int(row.subject), "window_idx": int(row.window_idx), "query": q,
                       "flag_if": bool(row.flag_if), "flag_lof": bool(row.flag_lof),
                       "sources": r["sources"], "tiers": r["tiers"],
                       "context": "\n\n---\n\n".join(f"[{s}]\n{d}" for s, d in zip(r["sources"], r["docs"]))})
    with open(p, "w", encoding="utf-8") as f:
        for a in alerts: f.write(json.dumps(a)+"\n")
    print(f"built + saved {len(alerts)} retrievals")
used = {s for a in alerts for s in a["sources"]}
t1 = sum(1 for a in alerts if any(t=="tier1" for t in a["tiers"]))
print(f"unique docs used: {len(used)} | alerts with >=1 guideline source: {t1}/{len(alerts)} ({100*t1/len(alerts):.1f}%)")

loaded 398 cached retrievals
unique docs used: 44 | alerts with >=1 guideline source: 70/398 (17.6%)


## 9. Labeled events — WESAD top-50 stress, MIT-BIH annotation-driven, PTB-XL stratified

In [9]:
# inter-patient constants (same as the detection notebook / THRESHOLDS.md)
DS1 = ["101","106","108","109","112","114","115","116","118","119","122","124",
       "201","203","205","207","208","209","215","220","223","230"]
DS2 = ["100","103","105","111","113","117","121","123","200","202","210","212",
       "213","214","219","221","222","228","231","232","233","234"]
MITBIH_DIR2 = ROOT/"Dataset/mit-bih-arrhythmia-database-1.0.0/mit-bih-arrhythmia-database-1.0.0"
AAMI2 = {"N","L","R","e","j"}
BEAT_SYMBOLS2 = AAMI2 | {"A","a","J","S","V","E","F","f","Q","/","!"}

# (a) WESAD: top-50 LOF-scored stress windows under the v1 pooled-baseline rule
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
Xb = np.vstack([w[f"X_{s}"][w[f"y_{s}"]==1] for s in w["subjects"]])
sc = StandardScaler().fit(Xb)
lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.15).fit(sc.transform(Xb))
cand = []
for s in w["subjects"]:
    s = int(s); Xs = w[f"X_{s}"][w[f"y_{s}"]==2]
    for i, scr in enumerate(-lof.score_samples(sc.transform(Xs))): cand.append((float(scr), s, i))
cand.sort(reverse=True); top_wes = cand[:50]
print(f"WESAD: {len(top_wes)} stress windows (top LOF scores {top_wes[0][0]:.2f}..{top_wes[-1][0]:.2f})")

# (b) MIT-BIH: annotation-driven selection (>=3 abnormal beats). The first-pass
#     detector-ranked selection chose 49/50 windows WITHOUT annotated ectopy —
#     superseded; see superseded_keys.json for the audit trail.
import wfdb
z_m = np.load(OUT/"cache/mitbih.npz", allow_pickle=True)
X_m2, y_m2, rec_m2 = z_m["X"], z_m["y"], z_m["record"].astype(str)
trm = np.isin(rec_m2, DS1) & (y_m2==0); tem = np.isin(rec_m2, DS2)
scaler_m = StandardScaler().fit(X_m2[trm])
lof_m = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.15).fit(scaler_m.transform(X_m2[trm]))
se_te = -lof_m.score_samples(scaler_m.transform(X_m2[tem]))
thr_m = np.percentile(-lof_m.score_samples(scaler_m.transform(X_m2[trm])), 85)
flag_abs = np.zeros(len(X_m2), bool); flag_abs[np.where(tem)[0][se_te > thr_m]] = True
cands = {"VEB": [], "SVEB": []}
for rec in DS2:
    sig, _ = wfdb.rdsamp(str(MITBIH_DIR2/rec))
    ann = wfdb.rdann(str(MITBIH_DIR2/rec), "atr")
    idx_te = np.where(tem & (rec_m2==rec))[0]
    seq = []
    for i, sym in zip(ann.sample, ann.symbol):
        if sym not in BEAT_SYMBOLS2: continue
        st_, en_ = i-144, i+144
        if st_ < 0 or en_ > sig.shape[0]: continue
        seq.append((i, sym, len(seq)))
    row_of = {pos: int(r) for pos, r in zip([x[2] for x in seq], idx_te)}
    wins = {}
    for sample, sym, pos in seq:
        wd = wins.setdefault(sample//(360*30), {"rows": [], "syms": []})
        wd["rows"].append(row_of[pos]); wd["syms"].append(sym)
    for widx, wd in wins.items():
        abn = [s_ for s_ in wd["syms"] if s_ not in AAMI2]
        if len(abn) < 3 or len(wd["rows"]) < 5: continue
        cls = "VEB" if any(s_ in {"V","E"} for s_ in abn) else "SVEB"
        cands[cls].append((len(abn), rec, widx, wd))
print(f"MIT-BIH candidate windows: VEB {len(cands['VEB'])}, SVEB {len(cands['SVEB'])} -> picked 25+25")

WESAD: 50 stress windows (top LOF scores 5.41..2.08)


MIT-BIH candidate windows: VEB 357, SVEB 101 -> picked 25+25


In [10]:
# (c) PTB-XL: stratified top-LOF fold-10 pathology per superclass (12 x 4 = 48)
zp = np.load(OUT/"cache/ptbxl.npz", allow_pickle=True)
Xp2, yp2, fp2 = zp["X"], zp["y"], zp["fold"]
trp = (fp2<=8) & (yp2==0)
sc_p = StandardScaler().fit(Xp2[trp])
lof_p = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.15).fit(sc_p.transform(Xp2[trp]))
se_fold10 = -lof_p.score_samples(sc_p.transform(Xp2[fp2==10]))
db = pd.read_csv(ROOT/"Dataset/ptb-xl-1.0.3/ptbxl_database.csv")
scp = pd.read_csv(ROOT/"Dataset/ptb-xl-1.0.3/scp_statements.csv", index_col=0)
def supers2(cp):
    import ast as _a
    out = set()
    for c in _a.literal_eval(cp):
        if c in scp.index and isinstance(scp.loc[c,"diagnostic_class"], str): out.add(scp.loc[c,"diagnostic_class"])
    return out
db["super"] = db.scp_codes.apply(supers2)
db["y"] = db.super.apply(lambda st: 0 if st == {"NORM"} else (-1 if not st else 1))
db = db[db.y != -1]          # same filter as the cache build — keeps row alignment
f10 = db[db.strat_fold==10].reset_index()
assert len(f10) == len(se_fold10), f"alignment: {len(f10)} vs {len(se_fold10)}"
score_by_ecg = {int(f10.iloc[k].ecg_id): float(se_fold10[k]) for k in range(len(f10))}
picked_ptbxl = []
for cls in ["MI","STTC","CD","HYP"]:
    cdb = f10[f10.super.apply(lambda st: cls in st)].copy()
    cdb["score"] = cdb.ecg_id.map(lambda e: score_by_ecg.get(e, -9.0))
    cdb = cdb.sort_values("score", ascending=False)
    for r in cdb.head(12).itertuples(): picked_ptbxl.append((r.ecg_id, r.filename_lr, cls))
print(f"PTB-XL: {len(picked_ptbxl)} records stratified 12x4:",
      dict(Counter(c for _,_,c in picked_ptbxl)))

PTB-XL: 48 records stratified 12x4: {'MI': 12, 'STTC': 12, 'CD': 12, 'HYP': 12}


## 10. Generation — strict grounded prompt; RAW text preserved; snap-logged canonicalizer

In [11]:
import ollama
SYSTEM_PROMPT_150 = """You are a clinical decision-support assistant that explains wearable biosignal anomalies.
You receive:
1. A description of an anomaly detected in a 30-second window of wearable signals.
2. Retrieved excerpts from peer-reviewed clinical guidelines and research articles.
STRICT RULES (never violate):
- Answer ONLY using the provided retrieved context.
- Cite the source document for every clinical claim. Format: [Source Name].
- If the retrieved context does not cover the anomaly, say: "The retrieved context is insufficient to explain this pattern."
- NEVER invent facts, numbers, citations, or medical conclusions not present in the context.
- This is a research tool, NOT a diagnostic device. State this once at the end.
- Keep the explanation under 150 words. Use plain language a nurse could understand.
Output format:
DETECTED: [one-sentence summary of what the anomaly pattern suggests]
EVIDENCE: [what the guidelines/literature say, with citations]
RECOMMENDATION: [what clinical follow-up the guidelines suggest, or "context insufficient"]
DISCLAIMER: Research decision-support tool. Not a diagnostic device. Does not replace clinical judgment."""

def canonicalize_fixed(raw_text, sources):
    """v1-intended canonicalizer with the ID capture FIXED: the citation ID is the PMC
    token; trailing slug text is display noise. v1's greedy regex dropped valid
    citations written as [PMC123_full_title]."""
    valid = sorted({s.split("_")[0] for s in sources if s.startswith("PMC")})
    snaps = []
    def _fix(m):
        cid = m.group(1)
        if cid in valid: return f"[{cid}]"
        close = difflib.get_close_matches(cid, valid, n=1, cutoff=0.75)
        if close:
            snaps.append({"before": cid, "after": close[0]}); return f"[{close[0]}]"
        snaps.append({"before": cid, "after": None}); return ""
    return re.sub(r"\[(PMC\d+)[^\]]*\]", _fix, raw_text), snaps

def generate_one(query, context, model=LLM_MODEL, prompt=SYSTEM_PROMPT_150, num_predict=500):
    resp = ollama.chat(model=model, think=False,
        messages=[{"role":"system","content":prompt},
                  {"role":"user","content":f"ANOMALY:\n{query}\n\nRETRIEVED CONTEXT:\n{context}"}],
        options={"temperature":0.1,"num_predict":num_predict,"num_ctx":10000,"num_gpu":99})
    return resp["message"]["content"].strip()

# live demonstration on two alerts
for a in alerts[:2]:
    raw = generate_one(a["query"], a["context"])
    fixed, snaps = canonicalize_fixed(raw, a["sources"])
    print(f"--- S{a['subject']} w{a['window_idx']} ({len(snaps)} citation repairs/drops)")
    print(fixed[:600], "\n")

--- S1 w1 (0 citation repairs/drops)
DETECTED: The abrupt, high-amplitude pattern in PPG signals suggests potential motion artifact rather than a true cardiac event like atrial fibrillation or tachypnea.

EVIDENCE: Guidelines indicate that abnormal readings from consumer heart rate devices (especially those using PPG technology) should be critically evaluated to distinguish suspected arrhythmia noise or oversensing caused by artifacts [ehra2022_digital_devices_arrhythmias]. Furthermore, inconclusive recordings and algorithm limitations remain barriers for wearable ECG/PPG devices in clinical use [PMC12731301].

RECOMMENDATION: Co 



--- S1 w6 (0 citation repairs/drops)
DETECTED: The abrupt high-amplitude signal suggests a respiratory event or arousal, potentially related to sleep-disordered breathing like tachypnea or apnea.
EVIDENCE: Changes in pulse rate and transit time can serve as markers of arousal from sleep [PMC13212801]. Wearable devices using oximetry and actigraphy are validated for detecting obstructive sleep apnea events, including desaturation indices that correlate with clinical severity [PMC12912879].
RECOMMENDATION: Review the patient's respiratory status; if this occurs during rest or sleep, consider evaluating for tachypnea or central/obst 



### Batch generation (resume-safe — loads the completed jsonl; the loop below is what a cold rebuild runs)

In [12]:
rows = [json.loads(l) for l in open(GEN_JSONL, encoding="utf-8")]
sup = json.loads((OUT/"superseded_keys.json").read_text())["superseded_mitbih_keys"]
rows = [r for r in rows if r["key"] not in sup]
print(f"generation_v2.jsonl: {len(rows)} explanations (50 superseded detector-ranked MIT-BIH keys excluded)")
print(df := pd.DataFrame([{k: r.get(k) for k in ("group","subgroup","model","prompt")} for r in rows]
        ).value_counts(["group","subgroup","model","prompt"]).to_frame("n"))
print(f"\ngeneration latency: {np.mean([r['latency_sec'] for r in rows if r['group']=='dalia' and r['subgroup']=='main']):.1f} s/alert mean")

generation_v2.jsonl: 646 explanations (50 superseded detector-ranked MIT-BIH keys excluded)
                                         n
group  subgroup    model       prompt     
dalia  main        qwen3.5:9b  150     398
       genablation llama3.1:8b 150      50
       wordcap     qwen3.5:9b  300      50
mitbih labeled     qwen3.5:9b  150      50
wesad  labeled     qwen3.5:9b  150      50
ptbxl  labeled     qwen3.5:9b  150      48

generation latency: 11.0 s/alert mean


## 11. Citation audit — raw vs repaired (deterministic, both texts preserved)

In [13]:
cit_re = re.compile(r"\[(PMC\d+)[^\]]*\]")
name_re = re.compile(r"\[([^]\[]+)\]")
stats = {"raw_cit":0, "raw_ok":0, "rep_cit":0, "rep_ok":0, "snaps":0, "drops":0,
         "t1_ok":0, "t1_bad":0}
for r in [x for x in rows if x["subgroup"]=="main"]:
    valid = {s.split("_")[0] for s in r["sources"] if s.startswith("PMC")}
    t1 = [s for s in r["sources"] if not s.startswith("PMC")]
    fixed, snaps = canonicalize_fixed(r["raw_explanation"], r["sources"])
    stats["snaps"] += sum(1 for s in snaps if s["after"])
    stats["drops"] += sum(1 for s in snaps if not s["after"])
    for which, txt in [("raw", r["raw_explanation"]), ("rep", fixed)]:
        cits = [m.group(1) for m in cit_re.finditer(txt)]
        stats[f"{which[:3]}_cit"] += len(cits)
        stats[f"{which[:3]}_ok"] += sum(1 for c in cits if c in valid)
    for m in name_re.finditer(fixed):
        tok = m.group(1).strip()
        if tok.startswith("PMC"): continue
        if any(s.startswith(tok) or tok.startswith(s) for s in t1): stats["t1_ok"] += 1
        else: stats["t1_bad"] += 1
print(f"RAW:      {stats['raw_cit']} citations, {stats['raw_ok']} valid = {100*stats['raw_ok']/stats['raw_cit']:.2f}%")
print(f"REPAIRED: {stats['rep_cit']} citations, {stats['rep_ok']} valid = {100*stats['rep_ok']/stats['rep_cit']:.2f}%")
print(f"canonicalizer: {stats['snaps']} snapped, {stats['drops']} dropped (genuine fabrications)")
print(f"guideline name-citations: {stats['t1_ok']} valid, {stats['t1_bad']} unmatched")

RAW:      1208 citations, 1196 valid = 99.01%
REPAIRED: 1196 citations, 1196 valid = 100.00%
canonicalizer: 0 snapped, 12 dropped (genuine fabrications)
guideline name-citations: 109 valid, 8 unmatched


## 12. Judge validation — corruption benchmark (v1's judge is a null instrument)

In [14]:
random_seed = 7
import random as _rnd; _rnd.seed(random_seed)
FABRICATED_FACTS = [
    "This pattern is associated with a 3-fold increase in 30-day mortality according to the sources.",
    "The literature states that patients with this finding require immediate hospital admission.",
    "According to the retrieved context, this signal pattern predicts stroke with 92% probability.",
    "The sources confirm that this finding always progresses to cardiac arrest if untreated.",
    "Retrieved guidelines recommend starting anticoagulation immediately for this pattern."]

main_rows = [r for r in rows if r["subgroup"]=="main"
             and len(re.findall(r"\[PMC\d+", r["explanation"])) >= 2]
sample = _rnd.sample(main_rows, 100)
bench = []
for i, r in enumerate(sample):
    txt, cits = r["explanation"], re.findall(r"PMC\d+", r["explanation"])
    ctype = i % 4; corrupted = txt
    if ctype == 0 and len(cits) >= 1:
        others = [s.split("_")[0] for s in r["sources"] if s.split("_")[0] not in cits[:1]]
        if others: corrupted = txt.replace(f"[{cits[0]}]", f"[{others[0]}]", 1)
        else: ctype = 1
    if ctype == 1:   corrupted = txt.replace("EVIDENCE:", f"EVIDENCE: {_rnd.choice(FABRICATED_FACTS)} ", 1)
    elif ctype == 2: corrupted = txt.replace(f"[{cits[0]}]", "[PMC99999999]", 1)
    elif ctype == 3: corrupted = txt.replace("DETECTED:", "DETECTED: This finding is diagnostic of acute myocardial infarction and requires emergency treatment. ", 1)
    bench.append({"row": r, "clean": txt, "corrupted": corrupted,
                  "ctype": ["citation_swap","fabricated_fact","fabricated_citation","diagnostic_exaggeration"][ctype]})
print("example corrupted item (type:", bench[0]["ctype"] + "):")
print(bench[0]["corrupted"][:280], "...")

example corrupted item (type: citation_swap):
DETECTED: The pattern suggests a physiological response to thermal stress or vasomotor change rather than respiratory failure, given elevated skin temperature alongside reduced respiration signal amplitude.

EVIDENCE: Wearable biosignals can be modified by local body temperature  ...


In [15]:
JUDGE_PROMPT = open(ROOT/"scripts/v2/judge_prompt_v2.txt", encoding="utf-8").read()

def local_judge(model, query, explanation, context):
    resp = ollama.chat(model=model, think=False,
        messages=[{"role":"system","content":JUDGE_PROMPT},
                  {"role":"user","content":f"QUERY: {query}\nSOURCES:\n{context}\nEXPLANATION:\n{explanation}"}],
        options={"temperature":0.1,"num_predict":300,"num_ctx":6000,"num_gpu":99})
    scores = {}
    for line in str(resp["message"]["content"]).split("\n"):
        m = re.match(r"(FAITHFULNESS|RELEVANCE|COMPLETENESS):\s*([123])", line.strip(), re.I)
        if m: scores[m.group(1).lower()] = int(m.group(2))
    return scores

# context resolver: generation rows store queries but not the chunk text (size);
# dalia contexts come from the cached retrieval, everything else re-retrieves
# deterministically (same corpus, same query, same embedder).
_alert_ctx = {f"S{a['subject']}|w{a['window_idx']}": a["context"] for a in alerts}
def ctx_of(row):
    k = row["key"].split("|", 1)[1].replace("mitbihv2|", "").replace("wesad|S", "S").replace("ptbxl|", "ptbxl|")
    if row["key"].startswith("dalia|"):
        return _alert_ctx[k]
    return retrieve(row["query"])["context"]

# live spot-check: gemma4 on 4 benchmark items (2 corrupted, 2 clean)
for b in bench[:2]:
    print("corrupted:", b["ctype"], "->", local_judge("gemma4:e4b", b["row"]["query"], b["corrupted"], ctx_of(b["row"])))
for b in bench[:2]:
    print("clean    :", local_judge("gemma4:e4b", b["row"]["query"], b["clean"], ctx_of(b["row"])))

# full validated results (200 calls per judge; loaded from the cached CSVs)
for model in JUDGE_CANDIDATES:
    d = pd.read_csv(OUT/f"judge_validation_{model.replace(':','_').replace('/','_')}.csv")
    det = (d[(d.is_corrupted==1) & (d.faithfulness==1)].shape[0]) / (d.is_corrupted==1).sum()
    fp  = (d[(d.is_corrupted==0) & (d.faithfulness==1)].shape[0]) / (d.is_corrupted==0).sum()
    by_type = d[d.is_corrupted==1].groupby("ctype").apply(lambda g: (g.faithfulness==1).mean(), include_groups=False).round(2).to_dict()
    print(f"{model}: detection {det:.2f}, FP {fp:.2f}, by type {by_type}")

corrupted: citation_swap -> {}


corrupted: fabricated_fact -> {'faithfulness': 1, 'relevance': 3, 'completeness': 2}


clean    : {}


clean    : {'faithfulness': 2, 'relevance': 3, 'completeness': 2}
llama3.1:8b: detection 0.00, FP 0.00, by type {'citation_swap': 0.0, 'diagnostic_exaggeration': 0.0, 'fabricated_citation': 0.0, 'fabricated_fact': 0.0}
gemma4:e4b: detection 0.48, FP 0.01, by type {'citation_swap': 0.0, 'diagnostic_exaggeration': 0.96, 'fabricated_citation': 0.16, 'fabricated_fact': 0.8}


## 13. Main judging (validated local judge) — distributions, not headlines

In [16]:
ev = pd.read_csv(OUT/"rag_evaluation_v2.csv")
agg = ev.groupby("subgroup").agg(n=("local_faithfulness","size"),
                                 faith=("local_faithfulness","mean"),
                                 relev=("local_relevance","mean"),
                                 compl=("local_completeness","mean")).round(2)
m = ev[(ev.group=="dalia") & (ev.subgroup=="main")]
print("dalia faithfulness distribution:", m.local_faithfulness.value_counts().sort_index().to_dict(),
      "(0 = parse failure)")
agg

dalia faithfulness distribution: {0: 45, 1: 2, 2: 176, 3: 175} (0 = parse failure)


,n,faith,relev,compl
subgroup,,,,
genablation,50,1.78,2.34,1.52
labeled,148,2.53,3.00,2.54
main,398,2.21,2.65,2.20
wordcap,50,2.02,2.52,2.12


## 14. Labeled-event concordance — labels never entered the queries

In [17]:
LEXICONS = {
 "stress":["stress","arousal","sympathetic","anxiety","mental load","psychological","emotional"],
 "VEB":["ventricular","pvc","premature ventricular","ventricular tachycard"],
 "SVEB":["supraventricular","atrial premature","pac","atrial ectopy","premature atrial",
         "atrial fibrillation","atrial tachyarrhythm"],
 "MI":["infarct","ischemi","stemi","coronary occlusion","st-elevation","st elevation"],
 "STTC":["repolarization","st depression","st-segment","st segment","t-wave","t wave inversion"],
 "CD":["conduction","bundle branch","heart block","av block","pr interval"],
 "HYP":["hypertroph","chamber enlargement","left ventricular mass"]}
ARTIFACT_TERMS = ["artifact","motion","sensor displacement","sensor contact","signal quality",
                  "electrode","noise","poor contact","device"]

def sec(t, a, b):
    mm = re.search(rf"{a}:\s*(.*?)(?={b}:|$)", t, re.S)
    return mm.group(1).strip().lower() if mm else ""

table = {}
for grp in ("wesad","mitbih","ptbxl"):
    sel = [r for r in rows if r["subgroup"]=="labeled" and r["group"]==grp]
    st = {"n":len(sel),"concordant":0,"artifact":0,"insufficient":0,"other":0,"by_label":{}}
    for r in sel:
        det = sec(r["explanation"], "DETECTED", "EVIDENCE"); lab = r["true_label"]
        st["by_label"].setdefault(lab, {"n":0,"ok":0}); st["by_label"][lab]["n"] += 1
        if any(t in det for t in LEXICONS.get(lab, [])):
            st["concordant"] += 1; st["by_label"][lab]["ok"] += 1
            if any(t in det for t in ARTIFACT_TERMS): st["artifact"] += 1
        elif "insufficient" in det: st["insufficient"] += 1
        elif any(t in det for t in ARTIFACT_TERMS): st["artifact"] += 1
        else: st["other"] += 1
    table[grp] = st
pd.DataFrame({g: {"n":v["n"], "concordant":f'{v["concordant"]} ({100*v["concordant"]/v["n"]:.0f}%)',
                  "artifact language":v["artifact"], "other":v["other"]} for g, v in table.items()}).T

,n,concordant,artifact language,other
wesad,50,47 (94%),28,1
mitbih,50,6 (12%),28,22
ptbxl,48,3 (6%),20,25


## 15. Before/after query-and-corpus fix — duplication & guideline reach

In [18]:
texts = [sec(r["explanation"],"DETECTED","EVIDENCE") + "\n" +
         sec(r["explanation"],"EVIDENCE","RECOMMENDATION")
         for r in rows if r["group"]=="dalia" and r["subgroup"]=="main"]
emb = embedder.encode(texts, normalize_embeddings=True)
sim = emb @ emb.T; np.fill_diagonal(sim, -1)
nn = sim.max(axis=1)
unassigned, cluster, cid = set(range(len(texts))), np.full(len(texts), -1), 0
while unassigned:
    seed = min(unassigned)
    members = [j for j in unassigned if sim[seed, j] > 0.9] + [seed]
    for j in members: cluster[j] = cid; unassigned.discard(j)
    cid += 1
nd1 = json.load(open(OUT/"rag_analysis_v1/near_duplicate_summary.json"))
print(pd.DataFrame({
  "v1": {"clusters@0.9": nd1["n_clusters_at_0.9"], "mean_NN_cos": nd1["mean_nn_cos"],
         "pct_twins>0.9": nd1["pct_rows_with_nn_gt_0.9"], "guideline_reach_%": 6.5},
  "v2": {"clusters@0.9": cid, "mean_NN_cos": round(float(nn.mean()),4),
         "pct_twins>0.9": round(float((nn>0.9).mean()*100),2), "guideline_reach_%": 17.6}}).T)

    clusters@0.9  mean_NN_cos  pct_twins>0.9  guideline_reach_%
v1         173.0       0.9356          81.66                6.5
v2         237.0       0.9149          67.59               17.6


## 16. Atomic-claim verification (FActScore-lite, different-family verifier)

In [19]:
fs = json.load(open(OUT/"factscore_lite.json"))
print({k: fs[k] for k in ("n_explanations","n_claims","pct_supported","pct_unsupported","pct_unverifiable","verifier")})
claims = pd.read_csv(OUT/"factscore_lite_claims.csv")
claims.sample(6, random_state=3)[["claim","verdict"]].to_string(index=False)

{'n_explanations': 60, 'n_claims': 797, 'pct_supported': 52.32, 'pct_unsupported': 0.0, 'pct_unverifiable': 47.68, 'verifier': 'gemma4:e4b (different family from generator)'}


'                                                                                                    claim      verdict\n                           The recommendation is to verify patient position (supine/sitting vs standing).    SUPPORTED\n     Recent physical activity should be considered before concluding elevated EDA indicates acute stress.    SUPPORTED\n                                                       The high standard deviation has a z-score of +9.2. UNVERIFIABLE\n                                                 The detected pattern suggests acute sympathetic arousal. UNVERIFIABLE\nThe anomaly suggests a sudden sympathetic stress response indicated by high electrodermal activity (EDA).    SUPPORTED\n                                                             The tool does not replace clinical judgment.    SUPPORTED'

## 17. Ablations — word cap & generator

In [20]:
for sg in ("main","wordcap","genablation"):
    sel = [r for r in rows if r["subgroup"]==sg and (sg!="main" or r["group"]=="dalia")]
    words = np.mean([len(r["explanation"].split()) for r in sel])
    s = ev[ev.subgroup==sg] if sg!="main" else ev[(ev.subgroup=="main") & (ev.group=="dalia")]
    print(f"{sg:12s} n={len(sel):3d} words={words:6.1f} faith={s.local_faithfulness.mean():.2f} compl={s.local_completeness.mean():.2f}")

main         n=398 words= 127.3 faith=2.21 compl=2.20
wordcap      n= 50 words= 196.5 faith=2.02 compl=2.12
genablation  n= 50 words=  99.1 faith=1.78 compl=1.52


## 18. Example alerts — one concordant, one failure mode

In [21]:
ex = next(r for r in rows if r["group"]=="wesad"
          and "stress" in sec(r["explanation"],"DETECTED","EVIDENCE")
          and "artifact" not in sec(r["explanation"],"DETECTED","EVIDENCE"))
print("=== CONCORDANT (true label:", ex["true_label"], ") ===")
print("QUERY:", ex["query"][:200])
print(ex["explanation"][:800])
print()
ptb = next(r for r in rows if r["group"]=="ptbxl")
print("=== TYPICAL PATHOLOGY FAILURE (true label:", ptb["true_label"], ") ===")
print("DETECTED:", sec(ptb["explanation"], "DETECTED", "EVIDENCE")[:400])

=== CONCORDANT (true label: stress ) ===
QUERY: Biosignal window flagged by anomaly detection. reduced bvp (z=-7.2 vs subject baseline) elevated wrist_eda (z=+4.0 vs subject baseline). Relevant topics: electrodermal activity skin conductance sympat
DETECTED: The pattern suggests acute sympathetic arousal (stress) where reduced blood volume pulse and elevated skin conductance indicate a physiological stress response common in anxiety or high cognitive load scenarios .

EVIDENCE: EDA reflects changes from sweat gland activity modulated by the autonomic nervous system, making it valuable for detecting emotional arousal and stress . When stressed, blood pressure increases causing higher heart rate linked with low HRV (reduced BVP) . EDA tends to increase during stressful periods while adding noise can affect PPG signals like the reduced BVP seen here .

RECOMMENDATION: Monitor for sustained elevation; consider multimodal confirmation if clinical context warrants, as single-signal methods h

## Reading

Under a validated judge: faithfulness 2.21/3 on wearable alerts (~44% fully faithful,
2 hallucination verdicts — the first non-zero count any judge has produced for this
system); 47.7% of atomic claims unverifiable from the retrieved context; concordance
94% (stress) / 12% (ectopy) / 6% (pathology ECG) with 42–56% of true pathology
attributed to artifact. Guideline reach tripled and duplication fell after the v2
query+corpus fixes, at the cost of narrower document spread (53→44). The clinician
kit (`clinician_eval/`) adjudicates these findings with human raters; the API-judge
columns await an OpenRouter key renewal (expired mid-run; handled gracefully).